In [1]:
# ============================================================
# LOGISTIC REGRESSION VISUALIZER
# Google Colab Version
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------

!pip install -q numpy matplotlib scikit-learn ipywidgets


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from sklearn.linear_model import LogisticRegression
from IPython.display import display, clear_output


# ============================================================
# LOGISTIC REGRESSION VISUALIZER
# ============================================================

class LogisticRegressionVisualizer:

    def __init__(self):

        # ----------------------------------------------------
        # DATA
        # ----------------------------------------------------

        self.X = []
        self.y = []

        # Trained model
        self.model = None

        # ----------------------------------------------------
        # TEST X SLIDER
        # ----------------------------------------------------

        self.test_slider = widgets.FloatSlider(
            value=5.0,
            min=0.0,
            max=10.0,
            step=0.1,
            description="Test X:",
            continuous_update=True,
            style={
                "description_width": "70px"
            },
            layout=widgets.Layout(
                width="400px"
            )
        )

        # ----------------------------------------------------
        # X INPUT
        # ----------------------------------------------------

        self.x_input = widgets.FloatText(
            value=3.0,
            description="X value:",
            style={
                "description_width": "70px"
            },
            layout=widgets.Layout(
                width="180px"
            )
        )

        # ----------------------------------------------------
        # BUTTONS
        # ----------------------------------------------------

        self.train_button = widgets.Button(
            description="Train Model",
            button_style="success",
            icon="check",
            layout=widgets.Layout(
                width="130px"
            )
        )

        self.example_button = widgets.Button(
            description="Load Example",
            button_style="info",
            layout=widgets.Layout(
                width="130px"
            )
        )

        self.reset_button = widgets.Button(
            description="Reset",
            button_style="warning",
            layout=widgets.Layout(
                width="100px"
            )
        )

        self.add_class0_button = widgets.Button(
            description="Add Class 0",
            layout=widgets.Layout(
                width="120px"
            )
        )

        self.add_class1_button = widgets.Button(
            description="Add Class 1",
            layout=widgets.Layout(
                width="120px"
            )
        )

        # ----------------------------------------------------
        # OUTPUT AREAS
        # ----------------------------------------------------

        self.plot_output = widgets.Output()

        self.info_output = widgets.Output()

        # ----------------------------------------------------
        # CONNECT BUTTONS
        # ----------------------------------------------------

        self.train_button.on_click(
            self.train_model
        )

        self.example_button.on_click(
            self.load_example
        )

        self.reset_button.on_click(
            self.reset
        )

        self.add_class0_button.on_click(
            self.add_class0
        )

        self.add_class1_button.on_click(
            self.add_class1
        )

        # ----------------------------------------------------
        # CONNECT SLIDER
        # ----------------------------------------------------

        self.test_slider.observe(
            self.update_test_point,
            names="value"
        )

        # ----------------------------------------------------
        # SHOW INTERFACE
        # ----------------------------------------------------

        self.show_interface()

        # Draw initial empty graph
        self.draw()


    # ========================================================
    # INTERFACE
    # ========================================================

    def show_interface(self):

        title = widgets.HTML(
            """
            <h1 style="margin-bottom:5px;">
                Logistic Regression Visualizer
            </h1>
            """
        )

        subtitle = widgets.HTML(
            """
            <p>
            Interactive demonstration of binary classification
            using Logistic Regression.
            </p>
            """
        )

        instructions = widgets.HTML(
            """
            <div style="
                background-color:#f5f5f5;
                padding:12px;
                border-radius:8px;
                margin-bottom:10px;
            ">

            <b>How to use:</b>

            <ol>
                <li>Click <b>Load Example</b>, OR add your own points.</li>
                <li>Class 0 = Negative Class.</li>
                <li>Class 1 = Positive Class.</li>
                <li>Click <b>Train Model</b>.</li>
                <li>Move the <b>Test X</b> slider.</li>
                <li>Observe probability and prediction.</li>
            </ol>

            </div>
            """
        )

        # ----------------------------------------------------
        # ADD DATA ROW
        # ----------------------------------------------------

        add_points = widgets.HBox(
            [
                self.x_input,
                self.add_class0_button,
                self.add_class1_button
            ],
            layout=widgets.Layout(
                margin="10px 0"
            )
        )

        # ----------------------------------------------------
        # MAIN BUTTON ROW
        # ----------------------------------------------------

        buttons = widgets.HBox(
            [
                self.example_button,
                self.train_button,
                self.reset_button
            ],
            layout=widgets.Layout(
                margin="10px 0"
            )
        )

        # ----------------------------------------------------
        # DISPLAY EVERYTHING
        # ----------------------------------------------------

        display(
            title,
            subtitle,
            instructions,
            widgets.HTML("<hr>"),
            widgets.HTML("<b>Add Training Data</b>"),
            add_points,
            buttons,
            widgets.HTML("<b>Test Point</b>"),
            self.test_slider,
            widgets.HTML("<br>"),
            self.info_output,
            self.plot_output
        )


    # ========================================================
    # ADD CLASS 0
    # ========================================================

    def add_class0(self, button=None):

        x = float(self.x_input.value)

        self.X.append([x])
        self.y.append(0)

        # Dataset changed, so old model is invalid
        self.model = None

        self.draw()


    # ========================================================
    # ADD CLASS 1
    # ========================================================

    def add_class1(self, button=None):

        x = float(self.x_input.value)

        self.X.append([x])
        self.y.append(1)

        # Dataset changed, so old model is invalid
        self.model = None

        self.draw()


    # ========================================================
    # LOAD EXAMPLE DATASET
    # ========================================================

    def load_example(self, button=None):

        self.X = [
            [1.0],
            [1.5],
            [2.0],
            [2.5],
            [3.0],
            [3.2],
            [3.5],
            [3.8],
            [4.0],
            [4.2],
            [4.5],
            [5.0],
            [5.5],
            [6.0],
            [6.5],
            [7.0],
            [7.5],
            [8.0],
            [8.5]
        ]

        self.y = [
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1
        ]

        # Reset model
        self.model = None

        self.draw()


    # ========================================================
    # TRAIN MODEL
    # ========================================================

    def train_model(self, button=None):

        # ----------------------------------------------------
        # CHECK NUMBER OF SAMPLES
        # ----------------------------------------------------

        if len(self.X) < 4:

            with self.info_output:

                clear_output(wait=True)

                print(
                    "Please add at least 4 data points."
                )

            return

        # ----------------------------------------------------
        # CHECK BOTH CLASSES
        # ----------------------------------------------------

        if len(set(self.y)) < 2:

            with self.info_output:

                clear_output(wait=True)

                print(
                    "You need both Class 0 and Class 1."
                )

            return

        # ----------------------------------------------------
        # CONVERT TO NUMPY ARRAYS
        # ----------------------------------------------------

        X_array = np.array(
            self.X,
            dtype=float
        )

        y_array = np.array(
            self.y,
            dtype=int
        )

        # ----------------------------------------------------
        # CREATE MODEL
        # ----------------------------------------------------

        self.model = LogisticRegression(
            solver="lbfgs"
        )

        # ----------------------------------------------------
        # TRAIN MODEL
        # ----------------------------------------------------

        self.model.fit(
            X_array,
            y_array
        )

        # ----------------------------------------------------
        # REDRAW
        # ----------------------------------------------------

        self.draw()


    # ========================================================
    # DRAW GRAPH
    # ========================================================

    def draw(self):

        with self.plot_output:

            clear_output(wait=True)

            # ------------------------------------------------
            # CREATE FIGURE
            # ------------------------------------------------

            fig, ax = plt.subplots(
                figsize=(12, 6)
            )

            # ------------------------------------------------
            # AXIS SETTINGS
            # ------------------------------------------------

            ax.set_xlim(
                0,
                10
            )

            ax.set_ylim(
                -0.1,
                1.1
            )

            ax.set_xlabel(
                "Feature X",
                fontsize=12
            )

            ax.set_ylabel(
                "Probability / Class",
                fontsize=12
            )

            ax.set_title(
                "Logistic Regression",
                fontsize=16,
                fontweight="bold"
            )

            ax.grid(
                alpha=0.3
            )

            # ------------------------------------------------
            # PLOT TRAINING DATA
            # ------------------------------------------------

            if len(self.X) > 0:

                X_array = np.array(
                    self.X,
                    dtype=float
                ).flatten()

                y_array = np.array(
                    self.y,
                    dtype=int
                )

                # Class 0
                class0 = X_array[
                    y_array == 0
                ]

                # Class 1
                class1 = X_array[
                    y_array == 1
                ]

                # --------------------------------------------
                # CLASS 0
                # --------------------------------------------

                if len(class0) > 0:

                    ax.scatter(
                        class0,
                        np.zeros(
                            len(class0)
                        ),
                        s=100,
                        marker="o",
                        label="Class 0"
                    )

                # --------------------------------------------
                # CLASS 1
                # --------------------------------------------

                if len(class1) > 0:

                    ax.scatter(
                        class1,
                        np.ones(
                            len(class1)
                        ),
                        s=100,
                        marker="o",
                        label="Class 1"
                    )

            # ------------------------------------------------
            # LOGISTIC REGRESSION CURVE
            # ------------------------------------------------

            if self.model is not None:

                # --------------------------------------------
                # Generate X values
                # --------------------------------------------

                x_values = np.linspace(
                    0,
                    10,
                    500
                ).reshape(
                    -1,
                    1
                )

                # --------------------------------------------
                # Calculate probabilities
                # --------------------------------------------

                probabilities = (
                    self.model
                    .predict_proba(
                        x_values
                    )[:, 1]
                )

                # --------------------------------------------
                # Plot sigmoid
                # --------------------------------------------

                ax.plot(
                    x_values,
                    probabilities,
                    linewidth=3,
                    label="P(Class 1)"
                )

                # --------------------------------------------
                # 0.5 THRESHOLD
                # --------------------------------------------

                ax.axhline(
                    0.5,
                    linestyle="--",
                    linewidth=1.5,
                    label="Threshold = 0.5"
                )

                # --------------------------------------------
                # GET COEFFICIENT
                # --------------------------------------------

                coefficient = (
                    self.model
                    .coef_[0][0]
                )

                # --------------------------------------------
                # GET INTERCEPT
                # --------------------------------------------

                intercept = (
                    self.model
                    .intercept_[0]
                )

                # --------------------------------------------
                # DECISION BOUNDARY
                # --------------------------------------------

                if abs(coefficient) > 1e-12:

                    boundary = (
                        -intercept /
                        coefficient
                    )

                    if 0 <= boundary <= 10:

                        ax.axvline(
                            boundary,
                            linestyle="--",
                            linewidth=2,
                            label=(
                                f"Decision Boundary = "
                                f"{boundary:.2f}"
                            )
                        )

                # --------------------------------------------
                # TEST POINT
                # --------------------------------------------

                test_x = float(
                    self.test_slider.value
                )

                # Probability of Class 1
                probability = (
                    self.model
                    .predict_proba(
                        [[test_x]]
                    )[0][1]
                )

                # Prediction
                prediction = (
                    self.model
                    .predict(
                        [[test_x]]
                    )[0]
                )

                # --------------------------------------------
                # PLOT TEST POINT
                # --------------------------------------------

                ax.scatter(
                    test_x,
                    probability,
                    s=250,
                    marker="*",
                    label="Test Point"
                )

                # --------------------------------------------
                # VERTICAL TEST LINE
                # --------------------------------------------

                ax.axvline(
                    test_x,
                    linestyle=":",
                    alpha=0.5
                )

            # ------------------------------------------------
            # LEGEND
            # ------------------------------------------------

            handles, labels = ax.get_legend_handles_labels()

            if handles:

                ax.legend(
                    loc="best"
                )

            # ------------------------------------------------
            # SHOW
            # ------------------------------------------------

            plt.tight_layout()

            plt.show()

        # ----------------------------------------------------
        # UPDATE INFORMATION
        # ----------------------------------------------------

        self.update_information()


    # ========================================================
    # UPDATE INFORMATION PANEL
    # ========================================================

    def update_information(self, change=None):

        with self.info_output:

            clear_output(wait=True)

            # ------------------------------------------------
            # NUMBER OF SAMPLES
            # ------------------------------------------------

            print(
                f"Training samples: {len(self.X)}"
            )

            # ------------------------------------------------
            # MODEL NOT TRAINED
            # ------------------------------------------------

            if self.model is None:

                print(
                    "\nModel not trained."
                )

                print(
                    "Load an example or add data, "
                    "then click 'Train Model'."
                )

                return

            # ------------------------------------------------
            # TEST X
            # ------------------------------------------------

            test_x = float(
                self.test_slider.value
            )

            # ------------------------------------------------
            # PROBABILITY
            # ------------------------------------------------

            probability = (
                self.model
                .predict_proba(
                    [[test_x]]
                )[0][1]
            )

            # ------------------------------------------------
            # PREDICTION
            # ------------------------------------------------

            prediction = (
                self.model
                .predict(
                    [[test_x]]
                )[0]
            )

            # ------------------------------------------------
            # COEFFICIENT
            # ------------------------------------------------

            coefficient = (
                self.model
                .coef_[0][0]
            )

            # ------------------------------------------------
            # INTERCEPT
            # ------------------------------------------------

            intercept = (
                self.model
                .intercept_[0]
            )

            # ------------------------------------------------
            # DECISION BOUNDARY
            # ------------------------------------------------

            if abs(coefficient) > 1e-12:

                boundary = (
                    -intercept /
                    coefficient
                )

            else:

                boundary = None

            # =================================================
            # DISPLAY MODEL INFORMATION
            # =================================================

            print(
                "\n========== MODEL =========="
            )

            print(
                f"Coefficient (β₁): "
                f"{coefficient:.4f}"
            )

            print(
                f"Intercept (β₀): "
                f"{intercept:.4f}"
            )

            print(
                "\nz = β₀ + β₁X"
            )

            print(
                f"z = {intercept:.4f} "
                f"+ ({coefficient:.4f})X"
            )

            print(
                "\nSigmoid Function:"
            )

            print(
                "P(Class 1) = 1 / (1 + e^(-z))"
            )

            # =================================================
            # TEST POINT INFORMATION
            # =================================================

            print(
                "\n========== TEST POINT =========="
            )

            print(
                f"X = {test_x:.1f}"
            )

            print(
                f"P(Class 1) = "
                f"{probability:.4f}"
            )

            print(
                f"P(Class 0) = "
                f"{1 - probability:.4f}"
            )

            # ------------------------------------------------
            # DECISION BOUNDARY
            # ------------------------------------------------

            if boundary is not None:

                print(
                    f"\nDecision Boundary = "
                    f"{boundary:.4f}"
                )

            else:

                print(
                    "\nDecision Boundary = Undefined"
                )

            # =================================================
            # PREDICTION
            # =================================================

            print(
                f"\nPrediction = CLASS "
                f"{prediction}"
            )

            # ------------------------------------------------
            # EXPLANATION
            # ------------------------------------------------

            if probability >= 0.5:

                print(
                    "\nP(Class 1) ≥ 0.5"
                )

                print(
                    "Therefore → CLASS 1"
                )

            else:

                print(
                    "\nP(Class 1) < 0.5"
                )

                print(
                    "Therefore → CLASS 0"
                )


    # ========================================================
    # UPDATE TEST POINT
    # ========================================================

    def update_test_point(self, change=None):

        if self.model is not None:

            self.draw()


    # ========================================================
    # RESET
    # ========================================================

    def reset(self, button=None):

        self.X = []
        self.y = []

        self.model = None

        self.draw()


# ============================================================
# START VISUALIZER
# ============================================================

visualizer = LogisticRegressionVisualizer()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.7 MB/s eta 0:00:00


HTML(value='\n            <h1 style="margin-bottom:5px;">\n                Logistic Regression Visualizer\n   …

HTML(value='\n            <p>\n            Interactive demonstration of binary classification\n            usi…

HTML(value='\n            <div style="\n                background-color:#f5f5f5;\n                padding:12p…

HTML(value='<hr>')

...

HTML(value='<b>Test Point</b>')

FloatSlider(value=5.0, description='Test X:', layout=Layout(width='400px'), max=10.0, style=SliderStyle(descri…

HTML(value='<br>')

Output()

Output()